# Exploring how a test is actually built

A working notebook, not a deliverable — for understanding the shape of Atom's assessments before trusting any conclusions built on top of them.

In [1]:
from google.cloud import bigquery
import pandas as pd

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 140)

client = bigquery.Client(project='atom-analytics-candidates')
D = 'atom-analytics-candidates.de_raw'

def q(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()

q(f'''
select 'responses' as table_name, count(*) as row_count from `{D}.responses`
union all select 'assessment_sittings', count(*) from `{D}.assessment_sittings`
union all select 'pupils', count(*) from `{D}.pupils`
union all select 'course_hierarchy', count(*) from `{D}.course_hierarchy`
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,row_count
0,responses,63707
1,assessment_sittings,2509
2,pupils,242
3,course_hierarchy,10552


## 1. Subjects in `course_hierarchy`

`subject_name` is the only human-readable label in this table. `topic_id` / `subtopic_id` / `atom_id` are just IDs — useful for counting and grouping, not for reading.

In [2]:
q(f'''
select subject_name, count(*) as questions,
       count(distinct topic_id) as topics,
       count(distinct subtopic_id) as subtopics,
       count(distinct atom_id) as atoms
from `{D}.course_hierarchy`
group by 1 order by 2 desc
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,subject_name,questions,topics,subtopics,atoms
0,English,4699,6,15,65
1,Maths,3593,7,21,86
2,Non-Verbal Reasoning,844,2,4,6
3,Verbal Reasoning,807,3,4,5
4,Science,335,4,16,37
5,Screeners,181,1,3,7
6,Wellbeing,93,1,3,6


## 2. The anatomy of a single test (a 'sitting')

Pick any `session_id` from `assessment_sittings` and look at every question answered in it

In [3]:
SESSION_ID = q(f'''
select session_id from `{D}.assessment_sittings`
where session_type = 'summative_assessment' and style = 'fixed_question' and is_complete
limit 10
''').session_id[9]

print('Looking at session:', SESSION_ID)

q(f'''
select r.question_number, r.question_id, h.subject_name, h.topic_id, h.subtopic_id,
       r.is_correct, r.is_no_attempt, r.seconds_taken, r.answered_at
from `{D}.responses` r
join `{D}.course_hierarchy` h using (question_id)
where r.session_id = '{SESSION_ID}'
order by r.question_number
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Looking at session: 7b380436-e9fa-40c9-81dc-c068143e0c7f


/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,question_number,question_id,subject_name,topic_id,subtopic_id,is_correct,is_no_attempt,seconds_taken,answered_at
0,1,c3ca038f-8289-4742-86f5-42e3e45c0aa3,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,f261df08-c8a4-49ad-8737-2c66af1d929d,False,False,108,2026-05-07 14:04:30+00:00
1,2,5b30a2bb-9b55-4745-8f33-2435e1386911,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,f261df08-c8a4-49ad-8737-2c66af1d929d,False,False,9,2026-05-07 14:04:40+00:00
2,3,7225edfc-3df4-4f63-8419-0a0712401b38,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,a606d770-d629-4e9c-88ad-debc53321c2e,False,False,24,2026-05-07 14:05:06+00:00
3,4,fca77d6f-c251-4221-88a1-39da894cb0fe,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,f261df08-c8a4-49ad-8737-2c66af1d929d,True,False,20,2026-05-07 14:05:27+00:00
4,5,503ec56a-7cbb-4ed1-8d0a-4fcbd2e80d91,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,f261df08-c8a4-49ad-8737-2c66af1d929d,False,False,33,2026-05-07 14:06:02+00:00
5,6,55f23655-0a38-4e8b-845f-7dd8cc81b7c4,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,a606d770-d629-4e9c-88ad-debc53321c2e,False,False,23,2026-05-07 14:06:26+00:00
6,7,024a0259-fe68-4f73-890e-aca2e9fe1a02,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,f261df08-c8a4-49ad-8737-2c66af1d929d,True,False,17,2026-05-07 14:06:47+00:00
7,8,b9970704-3743-40ed-8cc9-9d9c60e191b9,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,f261df08-c8a4-49ad-8737-2c66af1d929d,True,False,9,2026-05-07 14:07:02+00:00
8,9,2e4b3dfd-df9b-4e9d-877f-52a6a574531e,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,f261df08-c8a4-49ad-8737-2c66af1d929d,False,False,73,2026-05-07 14:08:17+00:00
9,10,4e13fb4f-d293-4176-86a3-f33376df84fb,English,a8116854-e75f-4d51-8f76-d97cfbe6a3df,a606d770-d629-4e9c-88ad-debc53321c2e,True,False,29,2026-05-07 14:08:47+00:00


## 3. Do English and Maths ever appear in the same test?

Every sitting I've found is single-subject. 
The cell below counts how many distinct subjects show up per `session_id` — if the design ever changes (or if there's an exception I've missed), this is the place that would catch it.

In [4]:
q(f'''
with subjects_per_sitting as (
    select r.session_id, count(distinct h.subject_name) as n_subjects,
           string_agg(distinct h.subject_name) as subjects
    from `{D}.responses` r
    join `{D}.course_hierarchy` h using (question_id)
    group by 1
)
select n_subjects, count(*) as sittings, array_agg(distinct subjects limit 5) as example_subject_lists
from subjects_per_sitting
group by 1 order by 1
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_subjects,sittings,example_subject_lists
0,1,2486,"[Maths, Screeners, English, Wellbeing, Non-Ver..."


### How broad is one sitting?

A sitting is always one subject — but how much of that subject does it cover? This is where the
Screeners give themselves away: every Screener sitting is exactly **one topic, one subtopic, one
atom**, roughly 19 questions deep on a single skill. English and Maths sittings range across 9 to 11
atoms. That's the difference between a battery of short, focused subtests and a normal test that
samples across the curriculum.

In [5]:
q(f'''
with per_sitting as (
    select r.session_id,
           any_value(h.subject_name) as subject_name,
           count(distinct h.topic_id) as topics,
           count(distinct h.subtopic_id) as subtopics,
           count(distinct h.atom_id) as atoms,
           count(*) as questions
    from `{D}.responses` r
    join `{D}.course_hierarchy` h using (question_id)
    group by 1
)
select subject_name, count(*) as sittings,
       round(avg(topics), 1) as avg_topics,
       round(avg(subtopics), 1) as avg_subtopics,
       round(avg(atoms), 1) as avg_atoms,
       round(avg(questions), 1) as avg_questions
from per_sitting group by 1 order by 2 desc
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,subject_name,sittings,avg_topics,avg_subtopics,avg_atoms,avg_questions
0,English,1060,1.7,3.3,8.8,26.9
1,Maths,707,2.0,4.9,11.0,27.2
2,Screeners,574,1.0,1.0,1.0,19.3
3,Wellbeing,129,1.0,3.0,6.0,31.3
4,Verbal Reasoning,13,1.6,1.6,2.0,53.2
5,Non-Verbal Reasoning,3,1.0,2.0,2.0,48.0


## 4. Screener questions — who takes them, and when

Screeners are a shared, single-topic subject sat across several year groups at once (see the main README). This section is for seeing that pattern directly rather than taking the README's word for it.

In [6]:
q(f'''
with sitting_subject as (
    select distinct r.session_id, h.subject_name
    from `{D}.responses` r join `{D}.course_hierarchy` h using (question_id)
)
select s.year_group_at_sitting as year_group, count(*) as sittings, count(distinct s.pupil_id) as pupils,
       min(s.started_at) as earliest, max(s.started_at) as latest
from `{D}.assessment_sittings` s
join sitting_subject ss using (session_id)
where ss.subject_name = 'Screeners'
group by 1 order by 1
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,year_group,sittings,pupils,earliest,latest
0,3,131,19,2026-06-02 10:50:29+00:00,2026-06-08 12:46:49+00:00
1,4,167,24,2026-06-11 08:52:22+00:00,2026-06-15 13:38:59+00:00
2,5,125,18,2026-06-10 13:42:29+00:00,2026-06-15 13:49:58+00:00
3,6,145,21,2026-06-12 08:13:47+00:00,2026-06-12 08:54:47+00:00


In [7]:
# When does each subject actually happen across the year?
# Comparing Screeners against every subject, not just one, shows the shape of the year:
# three termly English/Maths waves, then a separate June window for Screeners and Wellbeing.
q(f'''
with sitting_subject as (
    select distinct r.session_id, h.subject_name
    from `{D}.responses` r join `{D}.course_hierarchy` h using (question_id)
)
select format_timestamp('%Y-%m', s.started_at) as month, ss.subject_name, count(*) as sittings
from `{D}.assessment_sittings` s
join sitting_subject ss using (session_id)
where s.started_at is not null
group by 1, 2 order by 1, 2
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month,subject_name,sittings
0,2025-11,English,325
1,2025-11,Maths,213
2,2026-01,English,69
3,2026-01,Maths,40
4,2026-02,English,304
5,2026-02,Maths,215
6,2026-03,Maths,1
7,2026-05,English,292
8,2026-05,Maths,197
9,2026-05,Wellbeing,1


## 6. The small subjects — Verbal / Non-Verbal Reasoning, Science

Far fewer pupils than English or Maths, and the pattern suggests why.

Verbal and Non-Verbal Reasoning aren't national curriculum subjects — they're the classic **11+
entrance exam** subjects. They appear only in Y6, only in a two-day window in September, and their
papers run around 48 to 53 questions against roughly 27 for English and Maths. September of Y6 is
exactly when 11+ exams are sat. Every one of those 8 pupils is first seen in September 2026, with no
earlier history at all.

So the most plausible reading is a separate 11+ preparation cohort that has only just started, rather
than curriculum pupils doing a bit extra. The data supports the pattern; it can't confirm the reason.

**Science is a different case:** 335 questions defined in the hierarchy and zero responses, ever.
Content that exists but has never been used.

In [8]:
q(f'''
with sitting_subject as (
    select distinct r.session_id, h.subject_name
    from `{D}.responses` r join `{D}.course_hierarchy` h using (question_id)
)
select ss.subject_name, s.year_group_at_sitting as year_group,
       count(*) as sittings, count(distinct s.pupil_id) as pupils,
       min(s.started_at) as earliest, max(s.started_at) as latest
from `{D}.assessment_sittings` s
join sitting_subject ss using (session_id)
where ss.subject_name in ('Verbal Reasoning', 'Non-Verbal Reasoning', 'Science')
group by 1, 2 order by 1, 2
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,subject_name,year_group,sittings,pupils,earliest,latest
0,Non-Verbal Reasoning,6,3,3,2026-09-10 11:42:14+00:00,2026-09-10 11:42:17+00:00
1,Verbal Reasoning,6,13,8,2026-09-10 08:05:23+00:00,2026-09-11 10:07:21+00:00


In [9]:
# Are the Reasoning pupils a subset of the English/Maths pupils, or a separate group?
# (Science is excluded — it has zero responses, so it would only pad the numbers.)
q(f'''
with pupil_subject as (
    select distinct s.pupil_id, h.subject_name
    from `{D}.responses` r
    join `{D}.course_hierarchy` h using (question_id)
    join `{D}.assessment_sittings` s using (session_id)
),
reasoning as (
    select distinct pupil_id from pupil_subject
    where subject_name in ('Verbal Reasoning', 'Non-Verbal Reasoning')
),
english_maths as (
    select distinct pupil_id from pupil_subject
    where subject_name in ('English', 'Maths')
)
select
    (select count(*) from reasoning) as reasoning_pupils,
    (select count(*) from english_maths) as english_maths_pupils,
    (select count(*) from reasoning where pupil_id in (select pupil_id from english_maths))
        as pupils_doing_both
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,reasoning_pupils,english_maths_pupils,pupils_doing_both
0,8,156,0


## 7. Why are there ~10 sittings per pupil?

2,509 sittings across 242 pupils. Two things explain it, and neither is pupils resitting tests.

**A "test" is a short topic test, not a paper.** A full-year pupil sits roughly 6.7 different English
tests, 4.6 Maths, 6.9 Screener subtests and 1 Wellbeing survey — about 19 sittings, all of them
different tests.

**Nobody retakes anything.** For every subject, the number of *distinct tests* a pupil sat equals the
number of *sittings* exactly. There are 10 pupil+test pairs with more than one sitting, but all 10 of
those extra sittings have zero responses — empty records, not second attempts.

That matters for scoring: a pupil's percentage can never mix a first and second attempt at the same
questions, so pooling all their answers is safe.

In [16]:
# Distinct tests sat vs sittings. If these match, nobody is retaking anything.
q(f'''
with sitting_subject as (
    select distinct r.session_id, h.subject_name
    from `{D}.responses` r join `{D}.course_hierarchy` h using (question_id)
),
per_pupil as (
    select s.pupil_id, ss.subject_name,
           count(distinct s.assessment_id) as distinct_tests,
           count(*) as sittings
    from `{D}.assessment_sittings` s
    join sitting_subject ss using (session_id)
    group by 1, 2
)
select subject_name, count(*) as pupils,
       round(avg(distinct_tests), 1) as avg_distinct_tests,
       round(avg(sittings), 1) as avg_sittings,
       countif(sittings > distinct_tests) as pupils_with_a_repeat
from per_pupil group by 1 order by 2 desc
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,subject_name,pupils,avg_distinct_tests,avg_sittings,pupils_with_a_repeat
0,English,154,6.7,6.7,0
1,Maths,152,4.6,4.6,0
2,Wellbeing,125,1.0,1.0,0
3,Screeners,82,6.9,6.9,0
4,Verbal Reasoning,8,1.6,1.6,0
5,Non-Verbal Reasoning,3,1.0,1.0,0


### Two cohorts, not one

Sittings per pupil is bimodal — a cluster at 1–5, nothing at 6–9, then a cluster at 10–23. That gap
is the boundary between the established cohort (starting Nov 2025, a full year of testing behind
them) and the new intake that first appears in September 2026 with only a couple of weeks of
activity. Worth knowing before averaging anything across all pupils.

In [11]:
q(f'''
with per_pupil as (
    select pupil_id, count(*) as n_sittings, min(started_at) as first_seen
    from `{D}.assessment_sittings` group by 1
)
select format_timestamp('%Y-%m', first_seen) as first_seen_month,
       count(*) as pupils,
       round(avg(n_sittings), 1) as avg_sittings
from per_pupil group by 1 order by 1
''')

/Users/michaelamos/atom_learning/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,first_seen_month,pupils,avg_sittings
0,NaN,1,1.0
1,2025-11,113,18.7
2,2026-01,20,10.8
3,2026-02,1,11.0
4,2026-05,4,10.0
5,2026-08,1,1.0
6,2026-09,41,3.0


## Notes

Things worth writing down as you go, so this stays useful next time you open it:

- 
- 